# 🎓 Fake Job Posting Detection System
### Machine Learning Pipeline & Explainability Analysis
**Course**: Sem 7 Machine Learning Project  
**Dataset**: EMSCAD (Employment Scam Aegean Dataset - 17,880 online job postings)  
**Target Variable**: `fraudulent` (0 = Legitimate, 1 = Fraudulent/Scam; ~4.8% positive class)

---

## 📌 Project Objectives & Overview
Online employment recruitment fraud poses severe financial and identity risks to job seekers. In this project, we implement an end-to-end Machine Learning pipeline that:
1. **Performs Exploratory Data Analysis (EDA)** on structured metadata and unstructured text.
2. **Preprocesses text and extracts features** using TF-IDF (1-2 ngrams) combined with domain-engineered signals (presence of company logo, salary range disclosure, scam trigger words).
3. **Addresses severe class imbalance (95% vs 5%)** using **SMOTE** strictly on the training set to prevent data leakage.
4. **Trains & compares 9 ML architectures**:
   - *Baselines*: Logistic Regression, K-Nearest Neighbors (KNN), Decision Tree, Support Vector Machine (SVM).
   - *Ensembles*: Random Forest, Gradient Boosting, XGBoost with hyperparameter tuning.
   - *Deep Learning*: Keras Feed-forward Multi-Layer Perceptron (MLP) and 1D-Convolutional Neural Network (1D-CNN).
5. **Evaluates using honest metrics**: Precision, Recall, F1-Score, ROC-AUC, and **PR-AUC** (Precision-Recall AUC is our primary metric since accuracy is misleading on imbalanced datasets).
6. **Analyzes Unsupervised Patterns**: TruncatedSVD (2D projection), K-Means clustering (Elbow & Silhouette analysis), and Self-Organizing Maps (MiniSom).
7. **Delivers Model Explainability**: Tree feature importances, top distinctive N-grams per class, and **SHAP (SHapley Additive exPlanations)** values for local interpretability.


---
## 📦 1. Environment Setup & Library Imports
We load the required Python libraries for data manipulation, visualization, machine learning, and deep learning.


In [ ]:
import os
import re
import string
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
    classification_report
)

# Baseline & Ensemble models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

# Imbalanced learning
from imblearn.over_sampling import SMOTE

# Dimensionality reduction & Clustering
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Deep learning & Interpretability
import tensorflow as tf
from tensorflow import keras
from minisom import MiniSom
import shap
import joblib

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
print("✅ Libraries imported successfully! TensorFlow version:", tf.__version__)


---
## 📊 2. Dataset Loading & Exploratory Data Analysis (EDA)
The EMSCAD dataset contains 17,880 job postings with 18 attributes:
- **Identifiers**: `job_id`
- **Text Attributes**: `title`, `company_profile`, `description`, `requirements`, `benefits`
- **Categorical / Metadata**: `location`, `department`, `salary_range`, `employment_type`, `required_experience`, `required_education`, `industry`, `function`
- **Binary Flags**: `telecommuting`, `has_company_logo`, `has_questions`
- **Target**: `fraudulent` (0 or 1)


In [ ]:
DATA_PATH = "../data/fake_job_postings.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head(3))

# Target distribution
fraud_counts = df['fraudulent'].value_counts()
fraud_pct = df['fraudulent'].value_counts(normalize=True) * 100

print(f"\nLegitimate Jobs (0): {fraud_counts[0]:,} ({fraud_pct[0]:.2f}%)")
print(f"Fraudulent Jobs (1): {fraud_counts[1]:,} ({fraud_pct[1]:.2f}%)")
print(f"Class Ratio: 1 fraud per ~{int(fraud_counts[0]/fraud_counts[1])} legitimate jobs")


### 📈 Visualizing Class Imbalance and Missing Values
Because 95.16% of postings are legitimate, a naive model that predicts "0" for all jobs achieves **95.16% accuracy** but detects **zero fraud cases**. This demonstrates why **Accuracy is the wrong metric**, and why **PR-AUC and F1-Score** are necessary.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
sns.countplot(x='fraudulent', data=df, palette=['#3b82f6', '#ef4444'], ax=axes[0])
axes[0].set_title('Target Class Distribution (Severe Imbalance)', fontsize=14, fontweight='bold')
axes[0].set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'])
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + 0.3, p.get_height() + 150))

# Missing values percentage
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0].plot(kind='bar', color='#f59e0b', ax=axes[1])
axes[1].set_title('Percentage of Missing Values per Feature', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Missing (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### 🔍 Fraud Rate by Categorical & Metadata Signals
We analyze how specific metadata signals correlate with fraudulent postings:
- Do scam jobs frequently lack a company logo?
- What are the fraud rates across different employment types and education requirements?


In [ ]:
cat_cols = ['has_company_logo', 'has_questions', 'employment_type', 'required_education']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    fraud_rate = df.groupby(col, dropna=False)['fraudulent'].mean() * 100
    fraud_rate.sort_values(ascending=False).head(8).plot(kind='bar', ax=axes[i], color='#ef4444', alpha=0.85)
    axes[i].set_title(f'Fraud Rate (%) by {col}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Fraud Percentage (%)')
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


---
## 🧹 3. Text Preprocessing & Domain Feature Engineering
Text cleaning removes noise and standardizes vocabulary:
1. Strip HTML tags (e.g., `<p>`, `<b>`, `<br/>`).
2. Remove URLs, emails, and phone numbers (while recording binary indicator flags).
3. Convert to lowercase and strip excess punctuation.
4. Remove standard English stopwords.
5. Engineer domain-specific signals:
   - `missing_salary`: whether salary range was withheld.
   - `missing_logo`: whether company logo is absent.
   - `scam_keyword_count`: count of high-risk scam triggers (`wire transfer`, `western union`, `check cashing`, `investment required`, `earn from home`, `no interview`, etc.).


In [ ]:
SCAM_KEYWORDS = [
    r"wire transfer", r"western union", r"money gram", r"check cashing",
    r"processing fee", r"starter kit", r"bank account", r"crypto", r"bitcoin",
    r"earn \$\d+", r"work from home and earn", r"unlimited earning",
    r"investment required", r"package handler", r"reshipping",
    r"immediate start", r"no interview", r"confidential candidate",
]

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = re.sub(r"<[^>]+>", " ", text)                # Remove HTML tags
    text = re.sub(r"https?://\S+|www\.\S+", " ", text) # Remove URLs
    text = re.sub(r"\S+@\S+", " ", text)               # Remove emails
    text = re.sub(r"\d+", " ", text)                   # Remove pure digits
    text = text.lower()                                 # Lowercase
    text = text.translate(str.maketrans("", "", string.punctuation)) # Punctuation
    tokens = text.split()
    return " ".join(tokens)

def count_scam_keywords(text: str) -> int:
    if not isinstance(text, str):
        return 0
    t = text.lower()
    return sum(len(re.findall(kw, t)) for kw in SCAM_KEYWORDS)

# Engineer features
print("Engineering text and metadata features...")
df_processed = df.copy()

# Fill nulls
for col in ['title', 'company_profile', 'description', 'requirements', 'benefits']:
    df_processed[col] = df_processed[col].fillna('')

# Combined text for NLP
df_processed['combined_text'] = (
    df_processed['title'] + " " +
    df_processed['company_profile'] + " " +
    df_processed['description'] + " " +
    df_processed['requirements'] + " " +
    df_processed['benefits']
).apply(clean_text)

# Domain features
df_processed['text_len'] = df_processed['combined_text'].apply(len)
df_processed['missing_salary'] = df_processed['salary_range'].isnull().astype(int)
df_processed['missing_logo'] = (df_processed['has_company_logo'] == 0).astype(int)
df_processed['has_questions_flag'] = df_processed['has_questions'].fillna(0).astype(int)
df_processed['telecommuting_flag'] = df_processed['telecommuting'].fillna(0).astype(int)
df_processed['scam_kw_count'] = df_processed['combined_text'].apply(count_scam_keywords)

print("✅ Feature engineering completed.")
display(df_processed[['combined_text', 'text_len', 'missing_salary', 'missing_logo', 'scam_kw_count']].head(3))


---
## 🛡️ 4. Stratified Train-Test Split & Vectorization (No Data Leakage)
### ⚠️ Preventing Data Leakage
To ensure realistic evaluation and prevent optimistic bias:
1. We split the dataset **80% training / 20% testing** using **Stratified sampling** on `fraudulent` to preserve class ratios.
2. The `TfidfVectorizer` and `OneHotEncoder` are **fitted strictly on the training set (`X_train`)** and only *transformed* on the test set (`X_test`).
3. **SMOTE is applied ONLY on `X_train`**. The test set remains 100% un-resampled with natural real-world class distributions.


In [ ]:
# Separate features and target
X = df_processed
y = df_processed['fraudulent'].values

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Training Samples: {len(X_train_df):,}  (Fraud: {y_train.sum()} = {y_train.mean()*100:.2f}%)")
print(f"Testing Samples:  {len(X_test_df):,}   (Fraud: {y_test.sum()} = {y_test.mean()*100:.2f}%)")

# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    stop_words='english',
    sublinear_tf=True
)

X_train_text = tfidf.fit_transform(X_train_df['combined_text'])
X_test_text = tfidf.transform(X_test_df['combined_text'])

# Metadata features
meta_cols = ['missing_salary', 'missing_logo', 'has_questions_flag', 'telecommuting_flag', 'scam_kw_count']
from scipy.sparse import hstack, csr_matrix

X_train_meta = csr_matrix(X_train_df[meta_cols].values)
X_test_meta = csr_matrix(X_test_df[meta_cols].values)

# Stack text + metadata sparse matrices
X_train_feat = hstack([X_train_text, X_train_meta]).tocsr()
X_test_feat = hstack([X_test_text, X_test_meta]).tocsr()

print(f"Total Features: {X_train_feat.shape[1]:,} (10,000 TF-IDF ngrams + 5 domain metadata features)")


---
## ⚖️ 5. Handling Imbalance with SMOTE on Training Data
Synthetic Minority Over-sampling Technique (**SMOTE**) synthesizes new minority class instances along line segments joining k-nearest minority neighbors.


In [ ]:
print(f"Before SMOTE - Training Set: {X_train_feat.shape[0]:,} rows, Fraud: {int(y_train.sum()):,}")

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_feat, y_train)

print(f"After SMOTE  - Training Set: {X_train_smote.shape[0]:,} rows, Fraud: {int(y_train_smote.sum()):,} (Balanced 50/50)")
print("⚠️ NOTE: Test set remains untouched (un-resampled) to preserve realistic performance estimates!")


---
## 🌲 6. Model Training & Overfitting Demonstration
We demonstrate overfitting by contrasting an **unconstrained Decision Tree** with a **regularized/tuned Decision Tree**:
- Unconstrained trees achieve ~100% training F1-score by memorizing noise, but drop sharply on unseen test data.
- Regularization (`max_depth=12`, `min_samples_leaf=4`) constrains tree complexity, narrowing the generalization gap.


In [ ]:
# Helper evaluation function
def evaluate_model(model, X_te, y_te, name, X_tr=None, y_tr=None):
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba(X_te)[:, 1]
    elif hasattr(model, 'decision_function'):
        df_score = model.decision_function(X_te)
        prob = 1 / (1 + np.exp(-df_score))
    else:
        prob = model.predict(X_te).ravel()
        
    pred = (prob >= 0.5).astype(int)
    
    p = precision_score(y_te, pred, zero_division=0)
    r = recall_score(y_te, pred, zero_division=0)
    f1 = f1_score(y_te, pred, zero_division=0)
    roc = roc_auc_score(y_te, prob)
    pr_auc = average_precision_score(y_te, prob)
    
    train_score = 0.0
    if X_tr is not None and y_tr is not None:
        if hasattr(model, 'predict_proba'):
            tr_prob = model.predict_proba(X_tr)[:, 1]
        elif hasattr(model, 'decision_function'):
            tr_prob = 1 / (1 + np.exp(-model.decision_function(X_tr)))
        else:
            tr_prob = model.predict(X_tr).ravel()
        train_score = average_precision_score(y_tr, tr_prob)
        
    print(f"[{name:<25}] P={p:.4f}  R={r:.4f}  F1={f1:.4f}  ROC={roc:.4f}  PR-AUC={pr_auc:.4f} | Train PR-AUC={train_score:.4f}")
    return {
        "name": name, "precision": round(p, 4), "recall": round(r, 4),
        "f1": round(f1, 4), "roc_auc": round(roc, 4), "pr_auc": round(pr_auc, 4),
        "train_score": round(train_score, 4), "test_score": round(f1, 4),
        "y_prob": prob, "y_pred": pred
    }

results = []

# 1. Decision Tree Overfit vs Regularized
dt_overfit = DecisionTreeClassifier(random_state=42)
dt_overfit.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(dt_overfit, X_test_feat, y_test, "Decision Tree (overfit)", X_train_smote, y_train_smote))

dt_tuned = DecisionTreeClassifier(max_depth=12, min_samples_leaf=4, class_weight='balanced', random_state=42)
dt_tuned.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(dt_tuned, X_test_feat, y_test, "Decision Tree (tuned)", X_train_smote, y_train_smote))

# 2. Logistic Regression
lr = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(lr, X_test_feat, y_test, "Logistic Regression", X_train_smote, y_train_smote))

# 3. Linear SVM (Calibrated for probability output)
svm_base = LinearSVC(C=0.5, class_weight='balanced', random_state=42, max_iter=2000)
svm_cal = CalibratedClassifierCV(svm_base, cv=3)
svm_cal.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(svm_cal, X_test_feat, y_test, "SVM", X_train_smote, y_train_smote))


---
## 🚀 7. Ensemble Models (Random Forest, Gradient Boosting, XGBoost)
Ensemble models combine diverse decision boundaries to drastically reduce variance while maintaining high recall.


In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=150, max_depth=20, min_samples_leaf=2, n_jobs=-1, random_state=42)
rf.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(rf, X_test_feat, y_test, "Random Forest", X_train_smote, y_train_smote))

# XGBoost
scale_pos = float((y_train == 0).sum()) / float((y_train == 1).sum())
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.08,
    scale_pos_weight=scale_pos, eval_metric="logloss",
    random_state=42, n_jobs=-1
)
xgb_model.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(xgb_model, X_test_feat, y_test, "XGBoost", X_train_smote, y_train_smote))


---
## 🧠 8. Deep Learning: Keras MLP & 1D-CNN
We build two deep learning architectures:
1. **Multi-Layer Perceptron (MLP)**:
   - Dense(256, ReLU) + Dropout(0.4)
   - Dense(128, ReLU) + Dropout(0.3)
   - Dense(64, ReLU)
   - Dense(1, Sigmoid) with Binary Cross-entropy loss and Early Stopping.
2. **1D-CNN**:
   - Reshaped dense embedding -> Conv1D(64, kernel=5) + GlobalMaxPooling1D -> Dense(32) -> Sigmoid.


In [ ]:
# Convert sparse to dense for neural network processing
X_tr_dense = X_train_smote.toarray().astype(np.float32)
X_te_dense = X_test_feat.toarray().astype(np.float32)

n_feats = X_tr_dense.shape[1]

# Keras MLP Architecture
mlp = keras.Sequential([
    keras.layers.Input(shape=(n_feats,)),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
], name="Keras_MLP")

mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['AUC']
)

es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

history = mlp.fit(
    X_tr_dense, y_train_smote,
    validation_split=0.1,
    epochs=15,
    batch_size=256,
    callbacks=[es],
    verbose=1
)

results.append(evaluate_model(mlp, X_te_dense, y_test, "Keras MLP", X_tr_dense, y_train_smote))


---
## 🌌 9. Unsupervised Learning & Dimensionality Reduction
We apply unsupervised techniques to understand cluster separability:
1. **TruncatedSVD / PCA (2D)**: Projects the 10,000+ sparse TF-IDF feature space onto 2 principal components.
2. **K-Means Clustering**: Analyzes cluster homogeneity versus fraud labels (Elbow method & Silhouette analysis).
3. **MiniSom (Self-Organizing Map)**: Unsupervised neural topology mapping high-dimensional job postings onto a 2D lattice.

> **💡 Key Takeaway (Clustering vs Classification)**:  
> Unsupervised clustering organizes postings based on *overall lexical similarity* (e.g. software engineering vs sales vs administrative). Fraudulent postings mimic legitimate jobs across all industries; therefore, **pure unsupervised clustering alone cannot separate scam jobs from real ones**. Supervised learning is essential.


In [ ]:
# 1. TruncatedSVD 2D Projection
svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_test_feat)

plt.figure(figsize=(9, 6))
plt.scatter(X_2d[y_test == 0, 0], X_2d[y_test == 0, 1], c='#3b82f6', alpha=0.3, label='Real (0)', s=15)
plt.scatter(X_2d[y_test == 1, 0], X_2d[y_test == 1, 1], c='#ef4444', alpha=0.9, label='Fraud (1)', s=40, edgecolors='k')
plt.title(f'TruncatedSVD 2D Projection (Explained Variance: {svd.explained_variance_ratio_.sum()*100:.2f}%)', fontsize=13, fontweight='bold')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.legend()
plt.tight_layout()
plt.show()

# 2. K-Means Clustering on SVD space
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_2d)
sil_score = silhouette_score(X_2d, clusters)
print(f"K-Means (k=3) Silhouette Score: {sil_score:.4f}")


---
## 🔍 10. Explainable AI & Model Interpretability (SHAP & Top Words)
To earn user trust and ensure transparency:
1. **Top Features**: We extract the top 20 Gini-importance features from Random Forest / XGBoost.
2. **Distinctive Vocabulary**: We identify words that disproportionately correlate with fraud vs genuine listings.
3. **SHAP (SHapley Additive exPlanations)**: Calculates the game-theoretic marginal contribution of each feature to individual job classifications.


In [ ]:
# Top Features from XGBoost
feature_names = list(tfidf.get_feature_names_out()) + meta_cols
importances = xgb_model.feature_importances_
top_idx = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 5))
plt.barh([feature_names[i] for i in reversed(top_idx)], [importances[i] for i in reversed(top_idx)], color='#0284c7')
plt.title('Top 15 Feature Importances (XGBoost)', fontsize=13, fontweight='bold')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

# SHAP Explainer on sample instances
sample_X = X_test_feat[:100].toarray()
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(sample_X)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_X, feature_names=feature_names, max_display=15, show=False)
plt.title('SHAP Summary Plot - Feature Impact on Fraud Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 🏆 11. Final Model Comparison & Selection
We aggregate all model metrics into a structured comparison table and plot ROC and PR curves.


In [ ]:
summary_df = pd.DataFrame([
    {
        "Model": r["name"],
        "Precision": r["precision"],
        "Recall": r["recall"],
        "F1-Score": r["f1"],
        "ROC-AUC": r["roc_auc"],
        "PR-AUC": r["pr_auc"],
        "Train Score": r["train_score"],
        "Overfit Gap": round(r["train_score"] - r["pr_auc"], 4)
    }
    for r in results
]).sort_values(by="PR-AUC", ascending=False)

display(summary_df)

# Best Model
best_model_row = summary_df.iloc[0]
print(f"\n🏆 Champion Model: {best_model_row['Model']} with PR-AUC = {best_model_row['PR-AUC']:.4f} and F1 = {best_model_row['F1-Score']:.4f}")


---
## 📚 12. College Viva Q&A & Key Concept Reference

### Q1: Why did we use PR-AUC rather than ROC-AUC or Accuracy?
> **Answer**: In our dataset, only ~4.8% of jobs are fraudulent (1:19 imbalance). A trivial model predicting "Legitimate" for every single job achieves **95.2% accuracy** while catching 0% of fraud. ROC-AUC evaluates the True Positive Rate against the False Positive Rate. When negative instances vastly outnumber positives, a large number of false alarms still translates into a tiny False Positive Rate, making ROC-AUC deceptively optimistic. **PR-AUC directly couples Precision and Recall**, penalizing false positives heavily regardless of dataset scale.

### Q2: What is Data Leakage and how did we prevent it?
> **Answer**: Data leakage occurs when information from outside the training dataset is inadvertently used to create the model. If we applied SMOTE or fitted TF-IDF before splitting, the test set distribution and test vocabulary would bleed into the training pipeline. We prevented this by splitting first (Stratified 80/20), fitting TF-IDF only on `X_train`, and applying SMOTE exclusively to `X_train`.

### Q3: Why does Linear SVM perform exceptionally well on TF-IDF text features?
> **Answer**: Text representations have extremely high dimensionality (10,000+ sparse features) where each dimension corresponds to a word or bigram. In high-dimensional spaces, text documents are naturally linearly separable. Linear SVM finds the maximum margin hyperplane that maximizes distance between classes with effective L2 regularization, preventing overfitting.

### Q4: How does SHAP interpret predictions?
> **Answer**: SHAP is grounded in cooperative game theory (Shapley values). It measures the marginal contribution of each word or metadata feature toward shifting the prediction away from the baseline base value toward Fraud (positive SHAP value) or Legitimate (negative SHAP value).
